In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [ ]:
from src.config import get_path, get_sample_window

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

DATA_P4 = get_path("data_phase4")
RES_P4 = get_path("results_phase4")
FIG_DIR = RES_P4 / "figures"
TAB_DIR = RES_P4 / "tables"

# Load precomputed outputs
bkm = pd.read_parquet(DATA_P4 / "bkm_moments.parquet")
premia = pd.read_parquet(DATA_P4 / "cumulant_premia.parquet")
decomp_matched = pd.read_csv(TAB_DIR / "cyl_decomposition_matched.csv")  # headline
decomp_all = pd.read_csv(TAB_DIR / "cyl_decomposition.csv")              # supplementary
theta_rob = pd.read_csv(DATA_P4 / "theta_robustness.csv")
moment_summary = pd.read_csv(TAB_DIR / "moment_summary.csv", header=[0, 1], index_col=0)
bkm["date"] = pd.to_datetime(bkm["date"])
premia["date"] = pd.to_datetime(premia["date"])

bkm["date"] = pd.to_datetime(bkm["date"])
premia["date"] = pd.to_datetime(premia["date"])

print(f"BKM moments: {len(bkm)} day-venue pairs")
print(f"  CME: {(bkm['venue']=='CME').sum()} days")
print(f"  DER: {(bkm['venue']=='DER').sum()} days")
print(f"Cumulant premia: {len(premia)} rows")
print(f"  Date range: {premia['date'].min().date()} -> {premia['date'].max().date()}")

#### 1. Risk-Neutral Moment Summary

In [ ]:
display(moment_summary)

# Matched-day cross-venue correlations
cme = bkm[bkm["venue"] == "CME"].set_index("date")
der = bkm[bkm["venue"] == "DER"].set_index("date")
matched = cme.join(der, lsuffix="_cme", rsuffix="_der", how="inner")
print(f"\nMatched CME-DER days: {len(matched)}")
for col in ["var_Q", "skew_Q", "kurt_Q"]:
    rho = matched[[f"{col}_cme", f"{col}_der"]].corr().iloc[0, 1]
    print(f"  Cross-venue correlation ({col}): rho = {rho:.3f}")

#### 2. Risk-Neutral Moment Time Series

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for venue, color in [("CME", "C0"), ("DER", "C1")]:
    v = bkm[bkm["venue"] == venue].set_index("date").sort_index()
    axes[0].plot(v.index, v["var_Q"], color=color, lw=0.8, alpha=0.8, label=venue)
    axes[1].plot(v.index, v["skew_Q"], color=color, lw=0.8, alpha=0.8, label=venue)
    axes[2].plot(v.index, v["kurt_Q"], color=color, lw=0.8, alpha=0.8, label=venue)
axes[0].set_ylabel(r"$V_t^{\mathbb{Q},j}$")
axes[0].set_title("Risk-Neutral Moments at 27-day Horizon (BKM Extraction)")
axes[0].legend()
axes[1].set_ylabel(r"Skew$_t^{\mathbb{Q},j}$")
axes[1].axhline(0, color="black", lw=0.4)
axes[2].set_ylabel(r"Kurt$_t^{\mathbb{Q},j}$")
axes[2].axhline(3, color="black", lw=0.4, ls="--", label="Gaussian ref.")
axes[2].legend(loc="upper right")
axes[2].set_xlabel("Date")
plt.tight_layout()
plt.show()

#### 3. CL20 Lower-Bound Decomposition (Unconditional)

In [ ]:
print("=== HEADLINE: matched-day sample (thesis numbers) ===")
display(decomp_matched.round(4))
uncond = decomp_matched[decomp_matched["regime"] == "unconditional"]
for _, row in uncond.iterrows():
    print(f"  {row['venue']}: var {row['share_var']:.1%}, "
          f"skew {row['share_skew']:.1%}, kurt {row['share_kurt']:.1%} "
          f"| total 27d = {row['lb_total']:.4f} "
          f"(ann. {100*row['lb_total']*365/27:.0f}%)")

print("\n=== Supplementary: all available days per venue ===")
display(decomp_all[decomp_all["regime"] == "unconditional"].round(4))

In [ ]:
# Bar chart: unconditional lower-bound contributions
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(uncond)); width = 0.25
ax.bar(x - width, uncond["Pi_2"].values, width, label=r"$\Pi_2$ (variance)", color="C0")
ax.bar(x, uncond["Pi_3"].values, width, label=r"$\Pi_3$ (skewness)", color="C1")
ax.bar(x + width, uncond["Pi_4"].values, width, label=r"$\Pi_4$ (kurtosis)", color="C2")
ax.set_xticks(x); ax.set_xticklabels(uncond["venue"].values)
ax.axhline(0, color="black", lw=0.5)
ax.set_ylabel("27-day contribution")
ax.set_title("CL20 Lower-Bound Decomposition (Unconditional, θ = 2)")
ax.legend()
plt.tight_layout()
plt.show()

#### 4. CL24 Conditional Decomposition by Volatility Tercile

In [ ]:
# Tercile decomposition: stacked bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, venue in zip(axes, ["CME", "DER"]):
    t_data = decomp_matched[(decomp_matched["venue"] == venue) & (decomp_matched["regime"] != "unconditional")]
    terciles = ["low", "mid", "high"]
    x = np.arange(3)
    ax.bar(x, t_data.set_index("regime").loc[terciles, "Pi_2"].values,
           width=0.25, label=r"$\Pi_2$ var", color="C0")
    ax.bar(x + 0.25, t_data.set_index("regime").loc[terciles, "Pi_3"].values,
           width=0.25, label=r"$\Pi_3$ skew", color="C1")
    ax.bar(x + 0.5, t_data.set_index("regime").loc[terciles, "Pi_4"].values,
           width=0.25, label=r"$\Pi_4$ kurt", color="C2")
    ax.set_xticks(x + 0.25); ax.set_xticklabels(terciles)
    ax.set_title(venue); ax.legend(fontsize=9)
    ax.set_xlabel("Volatility tercile"); ax.set_ylabel("27-day contribution")
fig.suptitle("CL24 Conditional Decomposition by Volatility Tercile (θ = 2)", fontsize=13)
plt.tight_layout()
plt.show()

# Print shares by tercile
print("=== Moment shares by venue and tercile ===")
for venue in ["CME", "DER"]:
    print(f"\n  {venue}:")
    for regime in ["low", "mid", "high"]:
        row = decomp_matched[(decomp_matched["venue"] == venue) & (decomp_matched["regime"] == regime)].iloc[0]
        print(f"    {regime}: var {row['share_var']:.1%}, skew {row['share_skew']:.1%}, "
              f"kurt {row['share_kurt']:.1%} | total = {row['lb_total']:.4f}")

#### 5. Cumulant Contribution Time Series and Boxplots

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for venue, color in [("CME", "C0"), ("DER", "C1")]:
    v = premia[premia["venue"] == venue].set_index("date").sort_index()
    axes[0].plot(v.index, v["Pi_2"], color=color, lw=0.8, alpha=0.8, label=venue)
    axes[1].plot(v.index, v["Pi_3"], color=color, lw=0.8, alpha=0.8, label=venue)
    axes[2].plot(v.index, v["Pi_4"], color=color, lw=0.8, alpha=0.8, label=venue)
for ax in axes:
    ax.axhline(0, color="black", lw=0.4)
axes[0].set_ylabel(r"$\Pi_{2,t}^j$ (variance)")
axes[0].set_title("CL20 Cumulant Premium Contributions (θ = 2)")
axes[0].legend()
axes[1].set_ylabel(r"$\Pi_{3,t}^j$ (skewness)")
axes[2].set_ylabel(r"$\Pi_{4,t}^j$ (kurtosis)")
axes[2].set_xlabel("Date")
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots by venue x tercile
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (col, label) in zip(axes, [
    ("Pi_2", r"$\Pi_2$ (variance)"),
    ("Pi_3", r"$\Pi_3$ (skewness)"),
    ("Pi_4", r"$\Pi_4$ (kurtosis)")
]):
    data, labels_x = [], []
    for venue in ["CME", "DER"]:
        for tercile in ["low", "mid", "high"]:
            mask = (premia["venue"] == venue) & (premia["tercile"] == tercile)
            data.append(premia.loc[mask, col].dropna().values)
            labels_x.append(f"{venue}\n{tercile}")
    ax.boxplot(data, tick_labels=labels_x, showfliers=False)
    ax.axhline(0, color="black", lw=0.4)
    ax.set_ylabel(label); ax.tick_params(axis="x", labelsize=8)
fig.suptitle("Cumulant Premium Contributions by Venue and Volatility Tercile", fontsize=13)
plt.tight_layout()
plt.show()

#### 6. Cross-Venue Moment Agreement and VRP Diagnostic

In [ ]:
# Cross-venue scatter
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (col, label) in zip(axes, [
    ("var_Q", "Variance"), ("skew_Q", "Skewness"), ("kurt_Q", "Kurtosis")
]):
    ax.scatter(matched[f"{col}_cme"], matched[f"{col}_der"],
               s=3, alpha=0.4, color="C0")
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),
            max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, "k--", lw=0.5, alpha=0.5)
    ax.set_xlabel(f"CME {label}"); ax.set_ylabel(f"Deribit {label}")
    rho = matched[[f"{col}_cme", f"{col}_der"]].corr().iloc[0, 1]
    ax.set_title(f"{label} (ρ = {rho:.3f})")
fig.suptitle("Cross-Venue RN Moment Agreement (matched days)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# VRP time series
fig, ax = plt.subplots(figsize=(14, 5))
for venue, color in [("CME", "C0"), ("DER", "C1")]:
    v = premia[premia["venue"] == venue].set_index("date").sort_index()
    ax.plot(v.index, v["vrp"], color=color, lw=0.8, alpha=0.8, label=venue)
ax.axhline(0, color="black", lw=0.5)
ax.set_xlabel("Date")
ax.set_ylabel(r"$\mathrm{VRP}_t^j = V_t^{\mathbb{Q},j} - V_t^{\mathbb{P}}$")
ax.set_title("Variance Risk Premium: CME vs Deribit (27-day)")
ax.legend()
plt.tight_layout()
plt.show()

print("=== VRP Summary ===")
for venue in ["CME", "DER"]:
    v = premia[premia["venue"] == venue]["vrp"].dropna()
    print(f"  {venue}: mean = {v.mean():.4f}, std = {v.std():.4f}, "
          f"median = {v.median():.4f}, positive share = {(v > 0).mean():.1%}")

#### 7. Preference Parameter Robustness ($\theta$ sweep)

In [ ]:
print("=== Theta Robustness Sweep ===\n")
display(theta_rob.round(4))

fig, ax = plt.subplots(figsize=(10, 5))
for venue, color in [("CME", "C0"), ("DER", "C1")]:
    v = theta_rob[theta_rob["venue"] == venue].sort_values("theta")
    ax.plot(v["theta"], v["lb_annualized_pct"], "o-", color=color, label=venue, markersize=8)
ax.set_xlabel(r"Preference parameter $\theta$")
ax.set_ylabel("Annualized lower bound (%)")
ax.set_title("CL20 Lower Bound: Sensitivity to Preference Parameter")
ax.legend()
plt.tight_layout()
plt.show()

 ## Summary



 **Key findings (Phase 4, common window, $\theta = 2$):**



 1. **Kurtosis is co-equal with variance** in the CL20 lower bound (~30% vs ~52%

    unconditionally), distinguishing Bitcoin from the variance-dominated equity premium.



 2. **Composition shift across regimes**: variance share drops from ~67% (low-vol)

    to ~46% (high-vol) while kurtosis share rises from ~19% to ~35%, indicating

    that tail risk pricing intensifies under stress.



 3. **Cross-venue consistency**: Deribit exceeds CME at every moment order, but the

    *shares* are nearly identical (within 2 pp), indicating the MFK tent-shape

    operates uniformly across moment orders.



 4. **VRP diagnostic**: positive on both venues (CME 1.2%, DER 1.5% per 27 days),

    confirming the risk-neutral variance exceeds the physical variance.



 5. **Theta robustness**: the cross-venue ordering and composition shift are

    invariant to the preference parameter.

*(Note: the bullet numbers above are hard-coded prose and must be re-read against the post-rerun CSVs before quoting - the tables above are the source of truth.)*